# Lab 07 — Inference CLI Walkthrough

Runnable companion to `docs/15_inference_and_deployment_basics.md`. We
use the **already-trained Project 01 checkpoint** (`experiments/mlp_mnist/
checkpoints/best.pt`) and drive it three ways:

1. **As a Python API.** Load the checkpoint, build the preprocessing
   transform that matches training, predict on a single PIL image.
2. **As the CLI** in `src/inference.py` — exactly the command the
   end-user runs.
3. **Sanity-check** the predictions on five real MNIST PNGs (true
   label baked into the filename).

Prerequisite: you must have already run Project 01:

```bash
python src/train.py --config configs/mlp_mnist.yaml
```

If `experiments/mlp_mnist/checkpoints/best.pt` does not exist, this
notebook will tell you and stop.

Generated by `src/build_lab_07_inference_cli.py`.

## Setup

In [1]:
import json
import subprocess
import sys
from pathlib import Path as _Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torchvision import datasets

_ROOT = _Path.cwd()
while not (_ROOT / "ROADMAP.md").exists() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
print("repo root:", _ROOT)

CKPT = _ROOT / "experiments" / "mlp_mnist" / "checkpoints" / "best.pt"
SAMPLES_DIR = _ROOT / "datasets" / "mnist_samples_lab07"
MNIST_ROOT = str(_ROOT / "datasets" / "mnist")

if not CKPT.exists():
    raise FileNotFoundError(
        f"Could not find {CKPT}. Run `python src/train.py "
        f"--config configs/mlp_mnist.yaml` from the repo root first.")

sys.path.insert(0, str(_ROOT / "src"))
print("checkpoint:", CKPT)

repo root: /home/bangbc/Documents/AI_Courses/Deep_Learning_Foundation
checkpoint: /home/bangbc/Documents/AI_Courses/Deep_Learning_Foundation/experiments/mlp_mnist/checkpoints/best.pt


## Export 5 real MNIST test samples as PNGs

`torchvision.datasets.MNIST` returns PIL images; we save five test
images with the true label in the filename so we can verify each
prediction afterwards.

In [2]:
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)
mnist_test = datasets.MNIST(MNIST_ROOT, train=False, download=True, transform=None)

if not any(SAMPLES_DIR.glob("*.png")):
    for i, idx in enumerate([0, 1, 2, 3, 4]):
        img, label = mnist_test[idx]
        img.save(SAMPLES_DIR / f"sample_{i}_label{label}.png")

png_paths = sorted(SAMPLES_DIR.glob("*.png"))
print(f"sample PNGs ({len(png_paths)}):")
for p in png_paths:
    print(f"  {p.name}")

fig, axes = plt.subplots(1, len(png_paths), figsize=(11, 2))
for ax, p in zip(axes, png_paths):
    ax.imshow(np.array(Image.open(p)), cmap="gray"); ax.axis("off")
    ax.set_title(p.stem.split("_label")[-1], fontsize=10)
plt.tight_layout(); plt.show()

sample PNGs (5):
  sample_0_label7.png
  sample_1_label2.png
  sample_2_label1.png
  sample_3_label0.png
  sample_4_label4.png


/tmp/ipykernel_510478/323314866.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 1. Python-API path

Use the helpers in `src/inference.py` directly. Three things happen:

1. `torch.load(CKPT, weights_only=False)` reads the saved
   `model_state_dict` *plus* the config that produced it.
2. The model is rebuilt from that saved config (architecture stays in
   sync with the weights even if the source code drifts later).
3. The same preprocessing transform used at training time is bound to
   the model wrapper — this is the most common source of "my
   accuracy collapsed at inference" bugs.

In [3]:
from inference import load_for_inference, predict_one

model, transform, classes, cfg = load_for_inference(str(CKPT), device=torch.device("cpu"))
print(f"model     : {cfg['model']['name']}  ({sum(p.numel() for p in model.parameters()):,} params)")
print(f"classes   : {classes}")
print(f"transform : {transform}")
print(f"reported val_acc at save: {torch.load(CKPT, map_location='cpu', weights_only=False)['val_acc']:.4f}")

model     : mlp  (101,770 params)
classes   : ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
transform : Compose(
    Grayscale(num_output_channels=1)
    Resize(size=(28, 28), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=(0.1307,), std=(0.3081,))
)
reported val_acc at save: 0.9700


In [4]:
results_py = []
for p in png_paths:
    pil = Image.open(p).convert("RGB")
    cls, prob = predict_one(model, transform, classes, pil, device=torch.device("cpu"))
    truth = p.stem.split("_label")[-1]
    ok = (cls == truth)
    results_py.append({"path": p.name, "true": truth, "pred": cls, "prob": prob, "ok": ok})
    mark = "OK" if ok else "WRONG"
    print(f"  {p.name:40s} true={truth}  pred={cls}  prob={prob:.3f}  [{mark}]")
print()
print(f"Python API correct: {sum(r['ok'] for r in results_py)}/{len(results_py)}")

  sample_0_label7.png                      true=7  pred=7  prob=0.999  [OK]
  sample_1_label2.png                      true=2  pred=2  prob=0.998  [OK]
  sample_2_label1.png                      true=1  pred=1  prob=0.994  [OK]
  sample_3_label0.png                      true=0  pred=0  prob=1.000  [OK]
  sample_4_label4.png                      true=4  pred=4  prob=0.999  [OK]

Python API correct: 5/5


## 2. CLI path

Same model, same input, but invoked the way an end user would: a
shell command. We call `src/inference.py` via `subprocess` so the
notebook records the exact command line that ships with the project.

In [5]:
cli_out = _ROOT / "experiments" / "mlp_mnist" / "lab07_results.jsonl"
cli_out.parent.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(_ROOT / "src" / "inference.py"),
    "--checkpoint", str(CKPT),
    "--input",      str(SAMPLES_DIR),
    "--output",     str(cli_out),
    "--device",     "cpu",
]
print("$ " + " ".join(cmd))
print()
proc = subprocess.run(cmd, capture_output=True, text=True, cwd=str(_ROOT))
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:", proc.stderr)

$ /home/bangbc/miniforge3/envs/aicourse/bin/python3.11 /home/bangbc/Documents/AI_Courses/Deep_Learning_Foundation/src/inference.py --checkpoint /home/bangbc/Documents/AI_Courses/Deep_Learning_Foundation/experiments/mlp_mnist/checkpoints/best.pt --input /home/bangbc/Documents/AI_Courses/Deep_Learning_Foundation/datasets/mnist_samples_lab07 --output /home/bangbc/Documents/AI_Courses/Deep_Learning_Foundation/experiments/mlp_mnist/lab07_results.jsonl --device cpu



device: cpu, num_classes: 10
sample_0_label7.png                      -> 7  (0.999)
sample_1_label2.png                      -> 2  (0.998)
sample_2_label1.png                      -> 1  (0.994)
sample_3_label0.png                      -> 0  (1.000)
sample_4_label4.png                      -> 4  (0.999)

wrote 5 predictions to /home/bangbc/Documents/AI_Courses/Deep_Learning_Foundation/experiments/mlp_mnist/lab07_results.jsonl



In [6]:
# Parse what the CLI wrote and confirm it matches the Python API path.
results_cli = [json.loads(line) for line in cli_out.read_text().splitlines()]
print(f"CLI wrote {len(results_cli)} rows to {cli_out.name}\n")
print(f"{'path':<45s} {'pred':<6s} {'prob':<6s}")
for row in results_cli:
    print(f"  {_Path(row['path']).name:<45s} {row['class']:<6s} {row['prob']:.3f}")

# Cross-check against the Python API path.
cli_preds = {_Path(r['path']).name: r['class'] for r in results_cli}
py_preds  = {r['path']: r['pred']                for r in results_py}
matches   = sum(1 for k, v in py_preds.items() if cli_preds[k] == v)
print(f"\nPython API <-> CLI agreement: {matches}/{len(py_preds)}")

CLI wrote 5 rows to lab07_results.jsonl

path                                          pred   prob  
  sample_0_label7.png                           7      0.999
  sample_1_label2.png                           2      0.998
  sample_2_label1.png                           1      0.994
  sample_3_label0.png                           0      1.000
  sample_4_label4.png                           4      0.999

Python API <-> CLI agreement: 5/5


## 3. Sanity-check: predicted vs. true label

If every prediction matches the digit baked into the filename, the
inference pipeline is wired correctly end-to-end — preprocessing,
model architecture reconstruction, checkpoint load, softmax over
classes, file I/O.

In [7]:
truths = [p.stem.split("_label")[-1] for p in png_paths]
preds  = [results_cli[i]["class"]    for i in range(len(png_paths))]
correct = sum(t == p for t, p in zip(truths, preds))
print(f"CLI correct: {correct}/{len(png_paths)}")

fig, axes = plt.subplots(1, len(png_paths), figsize=(11, 2.5))
for ax, p, t, pred in zip(axes, png_paths, truths, preds):
    ax.imshow(np.array(Image.open(p)), cmap="gray"); ax.axis("off")
    tag = "OK" if t == pred else "WRONG"
    ax.set_title(f"true={t}  pred={pred}\n[{tag}]", fontsize=9)
plt.tight_layout(); plt.show()

CLI correct: 5/5


/tmp/ipykernel_510478/3641207372.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## What to copy into your own project's README

Every project should ship a "Inference" section that an end user can
copy-paste *exactly*:

```markdown
## Inference

Predict on a new image or directory using a trained checkpoint:

    python src/inference.py \
        --checkpoint experiments/<exp>/checkpoints/best.pt \
        --input  path/to/image_or_dir/ \
        --output results.jsonl

Output is JSONL with one prediction per line:

    {"path": "image.jpg", "class": "cat", "prob": 0.93}
```

If this block is missing or out of date, the project is not finished.